Script para encender (resume) o apagar (suspend) una capacidad de Microsoft Fabric
usando autenticación por Service Principal (OAuth2 client credentials flow).

In [1]:
import sys
import time
from typing import Optional

import requests

StatementMeta(, 47162631-af9a-4dcd-97e0-e28a8335c160, 3, Finished, Available, Finished, False)

In [ ]:
TENANT_ID = "TU_TENANT_ID"
CLIENT_ID = "TU_CLIENT_ID"
CLIENT_SECRET = "CAMBIAR_ESTE_VALOR"
SUBSCRIPTION_ID = "TU_SUBSCRIPTION_ID"
RESOURCE_GROUP = "TU_RESOURCE_GROUP"
CAPACITY_NAME = "TU_CAPACITY_NAME"

ACTION = "suspend"  # "resume" para encender, "suspend" para apagar


StatementMeta(, 47162631-af9a-4dcd-97e0-e28a8335c160, 4, Finished, Available, Finished, False)

In [3]:
# ==============================
# Constantes
# ==============================
ARM_SCOPE = "https://management.azure.com/.default"
TOKEN_URL_TEMPLATE = "https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
API_VERSION = "2023-11-01"
POLL_INTERVAL_SECONDS = 10
POLL_TIMEOUT_SECONDS = 1800  # 30 minutos


StatementMeta(, 47162631-af9a-4dcd-97e0-e28a8335c160, 5, Finished, Available, Finished, False)

In [4]:
class ScriptError(Exception):
    """Excepción controlada para errores del flujo del script."""


StatementMeta(, 47162631-af9a-4dcd-97e0-e28a8335c160, 6, Finished, Available, Finished, False)

In [5]:
def obtener_token(tenant_id: str, client_id: str, client_secret: str) -> str:
    """Obtiene un access token de Azure AD usando client credentials flow."""
    print("[INFO] Autenticando con Azure AD (client credentials)...")

    token_url = TOKEN_URL_TEMPLATE.format(tenant_id=tenant_id)
    payload = {
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": ARM_SCOPE,
        "grant_type": "client_credentials",
    }

    try:
        response = requests.post(token_url, data=payload, timeout=30)
    except requests.RequestException as exc:
        raise ScriptError(f"Error de red al solicitar token: {exc}") from exc

    if response.status_code != 200:
        detalle = response.text.strip()
        raise ScriptError(
            "Error de autenticación. "
            f"HTTP {response.status_code}. Respuesta: {detalle}"
        )

    try:
        token_data = response.json()
    except ValueError as exc:
        raise ScriptError("Respuesta de token no es JSON válido.") from exc

    access_token = token_data.get("access_token")
    if not access_token:
        raise ScriptError("No se recibió 'access_token' en la respuesta de autenticación.")

    print("[OK] Token obtenido correctamente.")
    return access_token

StatementMeta(, 47162631-af9a-4dcd-97e0-e28a8335c160, 7, Finished, Available, Finished, False)

In [6]:
def extraer_estado_desde_respuesta(response: requests.Response) -> Optional[str]:
    """Intenta extraer un estado de operación desde distintos formatos JSON de ARM."""
    try:
        data = response.json()
    except ValueError:
        return None

    if isinstance(data, dict):
        # Formatos comunes en ARM:
        # {"status": "InProgress|Succeeded|Failed"}
        # {"properties": {"provisioningState": "..."}}
        status = data.get("status")
        if isinstance(status, str):
            return status

        properties = data.get("properties")
        if isinstance(properties, dict):
            provisioning_state = properties.get("provisioningState")
            if isinstance(provisioning_state, str):
                return provisioning_state

    return None

StatementMeta(, 47162631-af9a-4dcd-97e0-e28a8335c160, 8, Finished, Available, Finished, False)

In [7]:
def hacer_polling(
    polling_url: str,
    headers: dict,
    intervalo_segundos: int = POLL_INTERVAL_SECONDS,
    timeout_segundos: int = POLL_TIMEOUT_SECONDS,
) -> None:
    """Realiza polling sobre la URL de operación asíncrona hasta finalizar."""
    print(f"[INFO] Operación asíncrona detectada. Iniciando polling")

    inicio = time.time()
    intento = 0

    while True:
        if (time.time() - inicio) > timeout_segundos:
            raise ScriptError(
                f"Timeout tras {timeout_segundos} segundos esperando la operación asíncrona."
            )

        intento += 1
        print(f"[INFO] Polling intento #{intento}...")

        try:
            response = requests.get(polling_url, headers=headers, timeout=30)
        except requests.RequestException as exc:
            raise ScriptError(f"Error de red durante polling: {exc}") from exc

        if response.status_code >= 400:
            detalle = response.text.strip()
            raise ScriptError(
                "Error HTTP durante polling. "
                f"HTTP {response.status_code}. Respuesta: {detalle}"
            )

        estado = extraer_estado_desde_respuesta(response)

        # ARM puede devolver 202 mientras sigue en curso.
        # Si no hay estado explícito y sigue 202, asumimos en progreso.
        if response.status_code == 202 and not estado:
            estado = "InProgress"

        if not estado:
            # En algunos casos Location puede acabar con 200 sin payload estándar.
            if response.status_code == 200:
                print("[OK] Polling finalizado con HTTP 200.")
                return

            print(
                "[WARN] No se pudo determinar estado en esta iteración; "
                f"HTTP {response.status_code}. Se reintenta..."
            )
            time.sleep(intervalo_segundos)
            continue

        estado_normalizado = estado.strip().lower()
        print(f"[INFO] Estado actual: {estado}")

        if estado_normalizado == "succeeded":
            print("[OK] Operación completada correctamente (Succeeded).")
            return

        if estado_normalizado == "failed":
            detalle = response.text.strip()
            raise ScriptError(f"La operación finalizó en Failed. Detalle: {detalle}")

        if estado_normalizado == "canceled":
            raise ScriptError("La operación fue cancelada (Canceled).")

        # Cualquier otro estado lo tratamos como en progreso.
        print(f"[INFO] Esperando {intervalo_segundos} segundos antes del siguiente polling...")
        time.sleep(intervalo_segundos)

StatementMeta(, 47162631-af9a-4dcd-97e0-e28a8335c160, 9, Finished, Available, Finished, False)

In [8]:
def ejecutar_accion(
    subscription_id: str,
    resource_group: str,
    capacity_name: str,
    action: str,
    token: str,
) -> None:
    """Ejecuta la acción resume/suspend sobre la capacidad de Fabric."""
    action_normalizada = action.strip().lower()
    if action_normalizada not in ("resume", "suspend"):
        raise ScriptError("ACTION debe ser 'resume' o 'suspend'.")

    endpoint = (
        f"https://management.azure.com/subscriptions/{subscription_id}"
        f"/resourceGroups/{resource_group}"
        f"/providers/Microsoft.Fabric/capacities/{capacity_name}/{action_normalizada}"
        f"?api-version={API_VERSION}"
    )

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    print(f"[INFO] Enviando petición '{action_normalizada}' a la capacidad '{capacity_name}'...")
    print(f"[DEBUG] Endpoint: {endpoint}")

    try:
        response = requests.post(endpoint, headers=headers, timeout=30)
    except requests.RequestException as exc:
        raise ScriptError(f"Error de red al ejecutar la acción '{action_normalizada}': {exc}") from exc

    if response.status_code == 200:
        print("[OK] Operación completada de forma inmediata (HTTP 200).")
        return

    if response.status_code == 202:
        polling_url = response.headers.get("Azure-AsyncOperation") or response.headers.get("Location")
        if not polling_url:
            raise ScriptError(
                "La API devolvió 202 (asíncrono), pero no incluyó cabecera "
                "'Azure-AsyncOperation' ni 'Location' para hacer polling."
            )

        hacer_polling(polling_url=polling_url, headers=headers)
        return

    detalle = response.text.strip()
    raise ScriptError(
        "Error al ejecutar acción sobre la capacidad. "
        f"HTTP {response.status_code}. Respuesta: {detalle}"
    )


StatementMeta(, 47162631-af9a-4dcd-97e0-e28a8335c160, 10, Finished, Available, Finished, False)

In [9]:
"""Punto de entrada principal del script."""
print("[INFO] Inicio de ejecución del script de control de capacidad Fabric.")

try:
    token = obtener_token(
        tenant_id=TENANT_ID,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    )

    ejecutar_accion(
        subscription_id=SUBSCRIPTION_ID,
        resource_group=RESOURCE_GROUP,
        capacity_name=CAPACITY_NAME,
        action=ACTION,
        token=token,
    )

    print("[OK] Proceso finalizado con éxito.")

except ScriptError as exc:
    print(f"[ERROR] {exc}")

except Exception as exc:  # Captura defensiva para errores inesperados.
    print(f"[ERROR] Error inesperado: {exc}")


StatementMeta(, 47162631-af9a-4dcd-97e0-e28a8335c160, 11, Finished, Available, Finished, False)

[INFO] Inicio de ejecución del script de control de capacidad Fabric.
[INFO] Autenticando con Azure AD (client credentials)...
[OK] Token obtenido correctamente.
[INFO] Enviando petición 'suspend' a la capacidad 'fbrbifabricpre'...
[DEBUG] Endpoint: https://management.azure.com/subscriptions/255d8a8d-fc9b-4d0d-b211-ed28e2518e1a/resourceGroups/rg-rbi-sql-data/providers/Microsoft.Fabric/capacities/fbrbifabricpre/suspend?api-version=2023-11-01
[INFO] Operación asíncrona detectada. Iniciando polling en: https://management.azure.com/subscriptions/255d8a8d-fc9b-4d0d-b211-ed28e2518e1a/providers/Microsoft.Fabric/locations/westeurope/operationstatuses/C44781F9-49C8-4AE1-ACBA-79C0B25000CE?api-version=2022-07-01-preview&t=639168497133621807&c=MIIHlTCCBn2gAwIBAgIRANClaz6RLmreUbR9r6zb9WMwDQYJKoZIhvcNAQELBQAwNjE0MDIGA1UEAxMrQ0NNRSBHMSBUTFMgUlNBIDIwNDggU0hBMjU2IDIwNDkgV1VTMiBDQSAwMTAeFw0yNjA0MTAwODE5MzhaFw0yNjEwMDUxNDE5MzhaMEAxPjA8BgNVBAMTNWFzeW5jb3BlcmF0aW9uc2lnbmluZ2NlcnRpZmljYXRlLm1hbmFnZW1lbnQuYX